# Experiment 7 - CNN for CIFAR-10 with Data Augmentation

**Aim:** Build a deeper convolutional neural network for natural-image
classification on CIFAR-10, using data augmentation and dropout to control
overfitting.

**Theory in brief:** CIFAR-10 contains 32x32 colour images of objects in ten
classes. Natural images vary in pose, lighting, background, and scale, so a
deeper network is needed to learn edges, textures, and object parts. Stacked
Conv2D blocks extract increasingly abstract features. Random horizontal flips
and translations enlarge the effective training set, while pooling and dropout
reduce overfitting on the 50,000 training images.

**Dataset:** CIFAR-10, downloaded at runtime from
`https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz`. The Python pickle
format is parsed directly so the data source and loader are explicit. No local
dataset is used.

**Expected result (original lab record):** about **76.76% test accuracy** with
a loss near **0.67** after 15 epochs.

**Runtime:** a GPU is strongly recommended. A full 15-epoch run can take a few
minutes on a Colab GPU and much longer on CPU. Set `FAST_DEV_RUN = True` for a
quick smoke test.

## 1. Environment setup

In [ ]:
import os

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
os.environ["PYTHONHASHSEED"] = "42"

import pickle
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix

SEED = 42
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print(f"TensorFlow : {tf.__version__}")
print(f"NumPy      : {np.__version__}")
print(f"GPU devices: {tf.config.list_physical_devices('GPU')}")

## 2. Download and parse CIFAR-10

CIFAR-10 is distributed as a tar archive of Python pickles. The loader:

1. downloads and extracts the archive with `tf.keras.utils.get_file`,
2. finds the `cifar-10-batches-py` directory (the exact return path of
   `get_file` differs between Keras versions),
3. unpickles the five training batches and the test batch,
4. reshapes each image from `(3072,)` to `(32, 32, 3)` and normalizes it.

In [ ]:
CIFAR10_URL = "https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz"


def find_cifar_root(path_like):
    path = Path(path_like)
    candidates = [
        path,
        path.parent,
        path / "cifar-10-batches-py",
        path.parent / "cifar-10-batches-py",
    ]
    for candidate in candidates:
        if (candidate / "data_batch_1").is_file():
            return candidate
    matches = sorted(path.parent.glob("**/data_batch_1"))
    if matches:
        return matches[0].parent
    raise FileNotFoundError(
        "Could not locate cifar-10-batches-py/data_batch_1. "
        f"Checked under: {path.parent}"
    )


def unpickle(file_path):
    with open(file_path, "rb") as file:
        return pickle.load(file, encoding="bytes")


extraction_path = tf.keras.utils.get_file(
    fname="cifar-10-python.tar.gz",
    origin=CIFAR10_URL,
    extract=True,
    cache_subdir="datasets",
)
CIFAR_ROOT = find_cifar_root(extraction_path)
print(f"get_file returned : {extraction_path}")
print(f"CIFAR root        : {CIFAR_ROOT}")


def load_cifar10(root):
    train_batches = [unpickle(root / f"data_batch_{index}") for index in range(1, 6)]
    test_batch = unpickle(root / "test_batch")
    meta = unpickle(root / "batches.meta")

    x_train = np.concatenate([batch[b"data"] for batch in train_batches], axis=0)
    y_train = np.concatenate(
        [np.asarray(batch[b"labels"], dtype=np.int64) for batch in train_batches],
        axis=0,
    )
    x_test = test_batch[b"data"]
    y_test = np.asarray(test_batch[b"labels"], dtype=np.int64)

    x_train = x_train.reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)
    x_test = x_test.reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)

    class_names = [
        name.decode("utf-8") if isinstance(name, bytes) else str(name)
        for name in meta[b"label_names"]
    ]
    return (
        (x_train.astype("float32") / 255.0, y_train),
        (x_test.astype("float32") / 255.0, y_test),
        class_names,
    )


(X_train, y_train), (X_test, y_test), CLASS_NAMES = load_cifar10(CIFAR_ROOT)

print(f"Training images: {X_train.shape}")
print(f"Test images    : {X_test.shape}")
print(f"Image range    : [{X_train.min():.2f}, {X_train.max():.2f}]")
print(f"Classes        : {CLASS_NAMES}")
print("Training class counts:", np.bincount(y_train).tolist())
print("Test class counts    :", np.bincount(y_test).tolist())

assert X_train.shape == (50000, 32, 32, 3)
assert X_test.shape == (10000, 32, 32, 3)
assert len(CLASS_NAMES) == 10

In [ ]:
fig, axes = plt.subplots(5, 5, figsize=(10, 10))
rng = np.random.default_rng(SEED)
sample_indices = rng.choice(len(X_train), size=25, replace=False)
for ax, index in zip(axes.ravel(), sample_indices):
    ax.imshow(X_train[index])
    ax.set_title(CLASS_NAMES[int(y_train[index])], fontsize=8)
    ax.axis("off")
fig.suptitle("Random CIFAR-10 training images", y=1.02)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "exp07_cifar_samples.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Data augmentation

The augmentation stack is part of the model, so it is active only during
training. Validation and test images are passed through unchanged.

In [ ]:
data_augmentation = tf.keras.Sequential(
    [
        tf.keras.layers.RandomFlip("horizontal", name="random_flip"),
        tf.keras.layers.RandomTranslation(0.1, 0.1, name="random_translation"),
    ],
    name="data_augmentation",
)

## 4. Model architecture

The network follows the lab sheet: three convolution blocks with 32, 64, and
128 filters, pooling and dropout after each block, then a 256-unit dense head
with dropout and a 10-way softmax.

In [ ]:
def build_cifar_cnn(seed=SEED):
    tf.keras.utils.set_random_seed(seed)
    model = tf.keras.Sequential(
        [
            tf.keras.layers.Input(shape=(32, 32, 3), name="image"),
            data_augmentation,
            tf.keras.layers.Conv2D(32, 3, activation="relu", padding="same", name="conv_1a"),
            tf.keras.layers.Conv2D(32, 3, activation="relu", padding="same", name="conv_1b"),
            tf.keras.layers.MaxPooling2D(name="pool_1"),
            tf.keras.layers.Dropout(0.25, name="dropout_1"),
            tf.keras.layers.Conv2D(64, 3, activation="relu", padding="same", name="conv_2a"),
            tf.keras.layers.Conv2D(64, 3, activation="relu", padding="same", name="conv_2b"),
            tf.keras.layers.MaxPooling2D(name="pool_2"),
            tf.keras.layers.Dropout(0.25, name="dropout_2"),
            tf.keras.layers.Conv2D(128, 3, activation="relu", padding="same", name="conv_3"),
            tf.keras.layers.MaxPooling2D(name="pool_3"),
            tf.keras.layers.Dropout(0.25, name="dropout_3"),
            tf.keras.layers.Flatten(name="flatten"),
            tf.keras.layers.Dense(256, activation="relu", name="dense_256"),
            tf.keras.layers.Dropout(0.5, name="dropout_4"),
            tf.keras.layers.Dense(10, activation="softmax", name="class_probabilities"),
        ],
        name="cifar10_cnn",
    )
    model.compile(
        optimizer=tf.keras.optimizers.Adam(),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


model = build_cifar_cnn()
model.summary()

## 5. Training configuration

`FAST_DEV_RUN` is provided so you can check that the notebook runs end-to-end
without waiting for the full 15 epochs. Leave it `False` for the reported
experiment.

In [ ]:
FAST_DEV_RUN = False
EPOCHS = 2 if FAST_DEV_RUN else 15
TRAIN_SAMPLES = 10000 if FAST_DEV_RUN else len(X_train)
BATCH_SIZE = 128

X_train_fit = X_train[:TRAIN_SAMPLES]
y_train_fit = y_train[:TRAIN_SAMPLES]

checkpoint_path = OUTPUT_DIR / "exp07_best_cifar_cnn.keras"
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(checkpoint_path),
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True,
        verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-5,
        verbose=1,
    ),
]

print(f"Fast dev run : {FAST_DEV_RUN}")
print(f"Epochs       : {EPOCHS}")
print(f"Training rows: {len(X_train_fit)}")
print(f"Batch size   : {BATCH_SIZE}")

## 6. Train the model

In [ ]:
start_time = time.perf_counter()
history = model.fit(
    X_train_fit,
    y_train_fit,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.1,
    verbose=2,
    callbacks=callbacks,
)
training_seconds = time.perf_counter() - start_time

print(f"Training finished in {training_seconds:.2f} seconds")
print(f"Best validation accuracy: {max(history.history['val_accuracy']):.4f}")
print(f"Checkpoint: {checkpoint_path.resolve()}")

## 7. Test evaluation

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
test_probabilities = model.predict(X_test, verbose=0)
test_predictions = test_probabilities.argmax(axis=1)

print(f"Test loss    : {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")
print()
print("Classification report:")
print(
    classification_report(
        y_test,
        test_predictions,
        target_names=CLASS_NAMES,
        digits=4,
        zero_division=0,
    )
)

In [ ]:
cm = confusion_matrix(y_test, test_predictions, labels=list(range(10)))
per_class_accuracy = cm.diagonal() / cm.sum(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(15, 6.5))

image = axes[0].imshow(cm, cmap="Blues")
axes[0].set_title("CIFAR-10 test confusion matrix")
axes[0].set_xlabel("Predicted class")
axes[0].set_ylabel("True class")
axes[0].set_xticks(range(10))
axes[0].set_yticks(range(10))
axes[0].set_xticklabels(CLASS_NAMES, rotation=45, ha="right")
axes[0].set_yticklabels(CLASS_NAMES)
fig.colorbar(image, ax=axes[0], fraction=0.046, pad=0.04)

axes[1].bar(range(10), per_class_accuracy, color="tab:blue")
axes[1].set_title("Per-class test accuracy")
axes[1].set_xlabel("Class")
axes[1].set_ylabel("Accuracy")
axes[1].set_ylim(0.0, 1.0)
axes[1].set_xticks(range(10))
axes[1].set_xticklabels(CLASS_NAMES, rotation=45, ha="right")
axes[1].grid(axis="y", alpha=0.3)

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "exp07_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

print("Per-class accuracy:")
for name, accuracy in zip(CLASS_NAMES, per_class_accuracy):
    print(f"  {name:12s}: {accuracy:.3f}")

## 8. Training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(history.history["loss"], label="Train", linewidth=2)
axes[0].plot(history.history["val_loss"], label="Validation", linewidth=2)
axes[0].set_title("Sparse categorical cross-entropy")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].grid(alpha=0.3)
axes[0].legend()

axes[1].plot(history.history["accuracy"], label="Train", linewidth=2)
axes[1].plot(history.history["val_accuracy"], label="Validation", linewidth=2)
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_ylim(0.0, 1.0)
axes[1].grid(alpha=0.3)
axes[1].legend()

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "exp07_training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. Sample predictions and misclassified images

In [ ]:
sample_indices = rng.choice(len(X_test), size=25, replace=False)
sample_probabilities = model.predict(X_test[sample_indices], verbose=0)
sample_predictions = sample_probabilities.argmax(axis=1)
sample_confidences = sample_probabilities.max(axis=1)

fig, axes = plt.subplots(5, 5, figsize=(12, 12))
for ax, index, prediction, confidence in zip(
    axes.ravel(), sample_indices, sample_predictions, sample_confidences
):
    ax.imshow(X_test[index])
    true_label = int(y_test[index])
    color = "green" if prediction == true_label else "red"
    ax.set_title(
        f"true {CLASS_NAMES[true_label]}\npred {CLASS_NAMES[int(prediction)]} ({confidence:.2f})",
        fontsize=8,
        color=color,
    )
    ax.axis("off")
fig.suptitle("Sample CIFAR-10 predictions", y=1.02)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "exp07_sample_predictions.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
misclassified_indices = np.flatnonzero(test_predictions != y_test)
print(f"Number of misclassified test images: {len(misclassified_indices)}")

if len(misclassified_indices) > 0:
    n_show = min(12, len(misclassified_indices))
    fig, axes = plt.subplots(3, 4, figsize=(12, 9))
    for ax, index in zip(axes.ravel(), misclassified_indices[:n_show]):
        ax.imshow(X_test[index])
        ax.set_title(
            f"true {CLASS_NAMES[int(y_test[index])]}\n"
            f"pred {CLASS_NAMES[int(test_predictions[index])]} "
            f"({test_probabilities[index].max():.2f})",
            fontsize=8,
            color="red",
        )
        ax.axis("off")
    for ax in axes.ravel()[n_show:]:
        ax.axis("off")
    fig.suptitle("Examples the CIFAR-10 CNN got wrong", y=1.02)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "exp07_misclassified.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("No misclassified test images.")

## 10. Inference on a single image

In [ ]:
def predict_cifar(image):
    image = np.asarray(image, dtype=np.float32)
    if image.shape != (32, 32, 3):
        raise ValueError(f"Expected a 32x32x3 image, got {image.shape}")
    if image.max() > 1.0:
        image = image / 255.0
    probabilities = model.predict(image.reshape(1, 32, 32, 3), verbose=0)[0]
    class_index = int(np.argmax(probabilities))
    return CLASS_NAMES[class_index], float(probabilities[class_index]), probabilities


sample_index = 0
predicted_class, confidence, all_probabilities = predict_cifar(X_test[sample_index])
print(f"True class : {CLASS_NAMES[int(y_test[sample_index])]}")
print(f"Prediction : {predicted_class} (confidence {confidence:.4f})")
print("Top 3 probabilities:")
for index in np.argsort(all_probabilities)[::-1][:3]:
    print(f"  {CLASS_NAMES[int(index)]:12s}: {all_probabilities[index]:.4f}")

## 11. Save model and metrics

In [ ]:
model_path = OUTPUT_DIR / "exp07_cifar10_cnn.keras"
model.save(model_path)

metrics = pd.DataFrame(
    [
        {
            "Test loss": float(test_loss),
            "Test accuracy": float(test_accuracy),
            "Training seconds": float(training_seconds),
            "Epochs run": len(history.history["loss"]),
        }
    ]
)
metrics.to_csv(OUTPUT_DIR / "exp07_metrics.csv", index=False)
pd.DataFrame(history.history).to_csv(OUTPUT_DIR / "exp07_history.csv", index=False)

print(f"Saved model to: {model_path.resolve()}")
print(metrics.to_string(index=False, float_format=lambda value: f"{value:.4f}"))

reloaded = tf.keras.models.load_model(model_path)
reloaded_predictions = reloaded.predict(X_test[:100], verbose=0).argmax(axis=1)
assert np.array_equal(reloaded_predictions, test_predictions[:100])
print("Reload check passed: first 100 predictions are identical.")

## 12. Conclusion

The augmented CNN is substantially stronger than a plain dense network on
CIFAR-10. Horizontal flips and small translations teach the model that object
identity is invariant to those transformations, and dropout reduces
overfitting in the deeper layers. The remaining errors are mostly
fine-grained confusions between visually similar classes (for example, cats
and dogs, or automobiles and trucks).

The original lab sheet records **76.76% test accuracy** and a loss near
**0.67** after 15 epochs. A rerun may differ because of GPU nondeterminism,
early stopping, and the validation split.

**Artifacts created at runtime (not committed):**

- `outputs/exp07_cifar10_cnn.keras`
- `outputs/exp07_best_cifar_cnn.keras`
- `outputs/exp07_metrics.csv`
- `outputs/exp07_history.csv`
- `outputs/exp07_cifar_samples.png`
- `outputs/exp07_confusion_matrix.png`
- `outputs/exp07_training_curves.png`
- `outputs/exp07_sample_predictions.png`
- `outputs/exp07_misclassified.png`

**Execution note:** the notebook was syntax-checked during repository
preparation; the full training run was not performed locally.